
# Manuscript Main Figures

This notebook renders the current main-paper figure set (`F1-F6`) from the frozen `P8` source tables.

Outputs are written to:

- `draft/figures/*.png`

This notebook does not rerun the analysis pipeline. It only converts frozen figure-source tables into draft-ready figures.


In [ ]:

from pathlib import Path
import os
import json
import math
import textwrap

ROOT = Path.cwd()
if not (ROOT / 'artifacts' / 'tables' / 'p8_F1_source.csv').exists():
    ROOT = ROOT.parent

TABLES = ROOT / 'artifacts' / 'tables'
DRAFT = ROOT / 'draft'
FIGURES = ROOT / 'artifacts' / 'draftV1' / 'figures'
RUNTIME = ROOT / 'artifacts' / 'runtime' / 'main_figure_notebook'
MPLCONFIG = RUNTIME / 'mplconfig'
XDG_CACHE = RUNTIME / 'xdg_cache'

for path in [FIGURES, RUNTIME, MPLCONFIG, XDG_CACHE]:
    path.mkdir(parents=True, exist_ok=True)

os.environ['MPLCONFIGDIR'] = str(MPLCONFIG)
os.environ['XDG_CACHE_HOME'] = str(XDG_CACHE)

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import seaborn as sns
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib.lines import Line2D
from IPython.display import display, Markdown

sns.set_theme(style='white', context='talk')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)

PIPELINE_VERSION = 'A1_GSTP2026Feb_contract_v1'
RENDER_RUN_ID = 'A1_GSTP2026Feb_contract_v1__20260321__figures_main_harvest__nogit'
SHAPEFILE = ROOT / 'data' / 'natural_earth' / 'ne_110m_admin_0_countries' / 'ne_110m_admin_0_countries.shp'

manifest = {
    'notebook': 'Manuscript_Main_Figures.ipynb',
    'pipeline_version': PIPELINE_VERSION,
    'render_run_id': RENDER_RUN_ID,
    'source_tables': [f'p8_F{i}_source.csv' for i in range(1, 7)],
    'output_dir': str(FIGURES),
}
(RUNTIME / 'manuscript_main_figures_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

display(Markdown(f"**Render run:** `{RENDER_RUN_ID}`"))
display(Markdown(f"**Figure output directory:** `{FIGURES}`"))



## Load frozen source tables and map support files


In [ ]:

F1 = pd.read_csv(TABLES / 'p8_F1_source.csv')
F2 = pd.read_csv(TABLES / 'p8_F2_source.csv')
F3 = pd.read_csv(TABLES / 'p8_F3_source.csv')
F4 = pd.read_csv(TABLES / 'p8_F4_source.csv')
F5 = pd.read_csv(TABLES / 'p8_F5_source.csv')
F6 = pd.read_csv(TABLES / 'p8_F6_source.csv')
T2 = pd.read_csv(TABLES / 'p8_T2_source.csv')
P1_BASELINE = pd.read_csv(TABLES / 'p1_retained_baseline.csv', low_memory=False)
P1_STATUS = pd.read_csv(TABLES / 'p1_status_scope_summary.csv')
P2_COUNT_RAW = pd.read_csv(TABLES / 'p2_country_mode_count_raw.csv', index_col=0)
P2_CAPACITY_RAW = pd.read_csv(TABLES / 'p2_country_mode_capacity_raw.csv', index_col=0)
P2_COUNT_NORM = pd.read_csv(TABLES / 'p2_country_mode_count_row_normalized.csv', index_col=0)
P2_CAPACITY_NORM = pd.read_csv(TABLES / 'p2_country_mode_capacity_row_normalized.csv', index_col=0)

WORLD = gpd.read_file(SHAPEFILE)
WORLD = WORLD.loc[WORLD['NAME'] != 'Antarctica'].copy()

country_points = (
    P1_BASELINE.assign(
        Latitude=pd.to_numeric(P1_BASELINE['Latitude'], errors='coerce'),
        Longitude=pd.to_numeric(P1_BASELINE['Longitude'], errors='coerce'),
    )
    .dropna(subset=['Latitude', 'Longitude'])
    .groupby('Country/Area', as_index=False)[['Latitude', 'Longitude']]
    .mean()
)

display(pd.DataFrame({
    'table': ['F1', 'F2', 'F3', 'F4', 'F5', 'F6'],
    'rows': [len(F1), len(F2), len(F3), len(F4), len(F5), len(F6)]
}))



## Helpers


In [ ]:

HARVEST = {
    'gray_blue': '#83A0BF',
    'pale_teal': '#BACED3',
    'primrose': '#EBD976',
    'gold': '#C8B015',
    'olive': '#ABC08D',
    'dark_grass': '#879E48',
}
FACE = '#ffffff'
CARD_FACE = '#ffffff'
TEXT_DARK = '#27313a'
GRID_SOFT = '#d9d3bb'
COUNT_VIEW_COLOR = HARVEST['gray_blue']
CAPACITY_VIEW_COLOR = HARVEST['dark_grass']
CAPACITY_COLORS = {
    1: HARVEST['dark_grass'],
    2: HARVEST['gray_blue'],
    3: HARVEST['gold'],
    4: HARVEST['olive'],
    5: HARVEST['primrose'],
    6: HARVEST['pale_teal'],
}
COUNT_COLORS = {
    1: HARVEST['dark_grass'],
    2: HARVEST['gray_blue'],
    3: HARVEST['gold'],
    4: HARVEST['pale_teal'],
    5: HARVEST['primrose'],
}
CLASS_COLORS = {
    'Discriminative in both': HARVEST['gold'],
    'Head-dominated in both': HARVEST['dark_grass'],
    'Count-only discriminative': HARVEST['gray_blue'],
    'Capacity-only discriminative': HARVEST['pale_teal'],
    'Background in both': HARVEST['olive'],
    'Mixed': HARVEST['primrose'],
}
MANUAL_NAME_FIXES = {
    'Türkiye': 'Turkey',
}
POINT_FALLBACKS = {'Singapore', 'Réunion'}
POINT_LABEL_OFFSETS = {
    'Singapore': (1.6, 1.0),
    'Réunion': (2.2, -1.1),
}


def save_figure(fig, stem, facecolor=FACE):
    png_path = FIGURES / f'{stem}.png'
    pdf_path = FIGURES / f'{stem}.pdf'
    if pdf_path.exists():
        pdf_path.unlink()
    fig.savefig(png_path, dpi=320, bbox_inches='tight', facecolor=facecolor)
    return str(png_path)


def clean_draft_name(text):
    if pd.isna(text):
        return ''
    return str(text).replace('Draft: ', '').strip()


def short_mode(mode):
    if pd.isna(mode):
        return ''
    return str(mode).replace('pre-construction', 'pre-constr.')


def format_mode_with_share(mode, share):
    if pd.isna(mode):
        return ''
    mode_text = short_mode(mode)
    if pd.notna(share):
        return f"{mode_text} ({float(share)*100:.0f}%)"
    return mode_text


def wrap_text(text, width=34):
    blocks = []
    for part in str(text).split('\n'):
        if not part.strip():
            blocks.append('')
            continue
        wrapped = textwrap.wrap(part, width=width) or ['']
        blocks.extend(wrapped)
    return '\n'.join(blocks)


def build_world_lookup(world):
    lookup = {}
    for idx, row in world.iterrows():
        for col in ['NAME', 'NAME_LONG', 'ADMIN', 'SOVEREIGNT', 'BRK_NAME', 'FORMAL_EN', 'NAME_EN']:
            value = row.get(col)
            if pd.notna(value):
                lookup.setdefault(str(value), idx)
    return lookup

WORLD_LOOKUP = build_world_lookup(WORLD)


def prepare_map_layers(country_df, country_col, group_col, palette):
    records = []
    unmatched = []
    for row in country_df[[country_col, group_col]].drop_duplicates().itertuples(index=False):
        country = row[0]
        group = row[1]
        lookup_name = MANUAL_NAME_FIXES.get(country, country)
        idx = WORLD_LOOKUP.get(lookup_name)
        if idx is None:
            unmatched.append((country, group))
        else:
            records.append({'world_idx': idx, 'Country/Area': country, group_col: group})

    matched_df = pd.DataFrame(records)
    world_plot = WORLD.copy()
    world_plot['world_idx'] = world_plot.index
    if len(matched_df):
        world_plot = world_plot.merge(matched_df, on='world_idx', how='left')
        world_plot['fill_color'] = world_plot[group_col].map(palette)
    else:
        world_plot['fill_color'] = None

    point_df = pd.DataFrame(unmatched, columns=['Country/Area', group_col])
    if len(point_df):
        point_df = point_df.merge(country_points, on='Country/Area', how='left')
        point_df['fill_color'] = point_df[group_col].map(palette)
    return world_plot, point_df


def draw_world_map(ax, world_plot, point_df, group_col, palette, ocean=FACE, land='#ffffff'):
    ax.set_facecolor(ocean)
    world_plot.plot(ax=ax, color=land, edgecolor='white', linewidth=0.35, zorder=1)
    colored = world_plot.loc[world_plot[group_col].notna()].copy()
    if len(colored):
        colored.plot(ax=ax, color=colored['fill_color'], edgecolor='white', linewidth=0.5, zorder=3)
    if len(point_df):
        ax.scatter(point_df['Longitude'], point_df['Latitude'], s=58, c=point_df['fill_color'], edgecolors=TEXT_DARK, linewidths=0.45, zorder=5)
        for row in point_df.itertuples(index=False):
            dx, dy = POINT_LABEL_OFFSETS.get(row[0], (1.5, 1.5))
            ax.text(row.Longitude + dx, row.Latitude + dy, row[0], fontsize=8.4, color=TEXT_DARK, zorder=6)
    ax.set_xlim(-180, 180)
    ax.set_ylim(-58, 88)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def add_card(ax, x, y, w, h, title, lines, color, face=CARD_FACE, title_color='#111111', title_fontsize=10.3, body_fontsize=8.8, border_width=2.2, body_offset=0.0):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.012,rounding_size=0.02',
                         linewidth=border_width, edgecolor=color, facecolor=face)
    ax.add_patch(box)

    title_width = max(18, int(w * 118))
    line_width = max(22, int(w * 154))
    title_parts = [str(part) for part in title] if isinstance(title, (list, tuple)) else [str(title)]
    title_parts = [part for part in title_parts if part.strip()]
    double_title = len(title_parts) >= 2
    if double_title:
        camp_text = wrap_text(title_parts[0], max(18, int(w * 128)))
        name_text = wrap_text(title_parts[1], max(18, int(w * 124)))
        camp_y = y + h - 0.030
        camp_rows = camp_text.count('\\n') + 1
        name_y = camp_y - 0.036 * camp_rows - 0.010
        name_rows = name_text.count('\\n') + 1
        ax.text(x + 0.016, camp_y, camp_text, fontsize=title_fontsize * 0.82, weight='bold', color=color, va='top', linespacing=1.00)
        ax.text(x + 0.016, name_y, name_text, fontsize=title_fontsize, weight='bold', color=title_color, va='top', linespacing=1.00)
        body_top = name_y - 0.050 * name_rows - 0.008
    else:
        title_text = wrap_text(title_parts[0], title_width)
        title_y = y + h - 0.030
        title_rows = title_text.count('\\n') + 1
        ax.text(x + 0.016, title_y, title_text, fontsize=title_fontsize, weight='bold', color=title_color, va='top', linespacing=1.01)
        body_top = title_y - 0.047 * title_rows - 0.008

    rendered_lines = [str(line) if 'Source communities:' in str(line) else wrap_text(line, line_width) for line in lines if str(line).strip()]
    if rendered_lines:
        anchor = body_top + body_offset
        for line_text in rendered_lines:
            ax.text(x + 0.016, anchor, line_text, fontsize=body_fontsize, color=TEXT_DARK, va='top', linespacing=1.02)
            line_rows = str(line_text).count('\\n') + 1
            anchor -= 0.056 * line_rows


def build_projection_from_matrix(df, similarity='cosine', top_k=6):
    values = df.to_numpy(dtype=float)
    countries = df.index.to_list()
    if similarity == 'cosine':
        row_norms = np.linalg.norm(values, axis=1, keepdims=True)
        row_norms[row_norms == 0] = 1.0
        sim = (values / row_norms) @ (values / row_norms).T
    elif similarity == 'dot':
        sim = values @ values.T
    else:
        raise ValueError(f'Unsupported similarity: {similarity}')
    np.fill_diagonal(sim, 0.0)
    graph = nx.Graph()
    graph.add_nodes_from(countries)
    for i, country in enumerate(countries):
        weights = sim[i].copy()
        if top_k < len(countries):
            candidate_idx = np.argpartition(weights, -top_k)[-top_k:]
        else:
            candidate_idx = np.arange(len(countries))
        ranked_idx = sorted(candidate_idx, key=lambda j: weights[j], reverse=True)
        for j in ranked_idx:
            if i == j or weights[j] <= 0:
                continue
            other = countries[j]
            weight = float(weights[j])
            if graph.has_edge(country, other):
                graph[country][other]['weight'] = max(graph[country][other]['weight'], weight)
            else:
                graph.add_edge(country, other, weight=weight)
    return graph


def compute_spread_layout(graph, strengths, seed=42):
    base_pos = nx.spring_layout(graph, seed=seed, weight='weight', k=1.45, iterations=500)
    nodes = list(graph.nodes())
    pos = {node: np.array(base_pos[node], dtype=float) for node in nodes}
    max_strength = max(strengths.values()) if strengths else 1.0
    radii = {node: 0.042 + 0.070 * np.sqrt(strengths.get(node, 0.0) / max_strength) for node in nodes}
    for _ in range(220):
        moved = False
        for i, node_i in enumerate(nodes):
            for node_j in nodes[i + 1:]:
                delta = pos[node_j] - pos[node_i]
                dist = float(np.linalg.norm(delta))
                min_dist = radii[node_i] + radii[node_j] + 0.030
                if dist < 1e-9:
                    angle = (i + 1) * 0.37
                    delta = np.array([np.cos(angle), np.sin(angle)])
                    dist = 1.0
                if dist < min_dist:
                    direction = delta / dist
                    shift = (min_dist - dist) * 0.52
                    pos[node_i] -= direction * shift
                    pos[node_j] += direction * shift
                    moved = True
        if not moved:
            break
    coords = np.array(list(pos.values()))
    center = coords.mean(axis=0)
    for node in nodes:
        pos[node] = pos[node] - center
    coords = np.array(list(pos.values()))
    max_abs = np.abs(coords).max()
    if max_abs > 0:
        for node in nodes:
            pos[node] = pos[node] / max_abs
    return pos


def combined_mode_class(row):
    if bool(row['discriminative_in_both']):
        return 'Discriminative in both'
    if bool(row['head_dominated_in_both']):
        return 'Head-dominated in both'
    if bool(row['count_only_discriminative']):
        return 'Count-only discriminative'
    if bool(row['capacity_only_discriminative']):
        return 'Capacity-only discriminative'
    if row['mode_class_count'] == 'background' and row['mode_class_capacity'] == 'background':
        return 'Background in both'
    return 'Mixed'



## Figure 1 — workflow and retained scope


In [ ]:

workflow = F1.loc[F1['section'] == 'workflow'].sort_values('display_order').copy()
status_retained = P1_STATUS.loc[P1_STATUS['status_group'] == 'retained'].copy().sort_values('rows_utility_sheet', ascending=False)

countries = int(F1.loc[F1['item_key'] == 'countries_retained', 'item_value'].iloc[0])
rows = int(float(F1.loc[F1['item_key'] == 'rows_retained', 'item_value'].iloc[0]))
capacity_tw = float(F1.loc[F1['item_key'] == 'capacity_mw_retained', 'item_value'].iloc[0]) / 1_000_000
pre_filter_rows = int(float(F1.loc[F1['item_key'] == 'rows_after_status_and_capacity_filters', 'item_value'].iloc[0]))

fig = plt.figure(figsize=(15.0, 8.35), facecolor=FACE)
outer = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1.32, 1.0], wspace=0.16)
ax_flow = fig.add_subplot(outer[0, 0])
right = outer[0, 1].subgridspec(2, 1, height_ratios=[1.12, 0.88], hspace=0.14)
ax_cards = fig.add_subplot(right[0, 0])
ax_status = fig.add_subplot(right[1, 0])

ax_flow.set_facecolor(FACE)
ax_flow.set_axis_off()
ax_flow.set_xlim(0, 1)
ax_flow.set_ylim(0, 1)

box_x = 0.045
box_w = 0.80
box_h = 0.075
top_edge = 0.93
bottom_edge = 0.09
y_positions = np.linspace(top_edge - box_h, bottom_edge, len(workflow))
box_colors = [HARVEST['pale_teal'], HARVEST['olive'], HARVEST['primrose'], HARVEST['gold'], HARVEST['gray_blue'], HARVEST['pale_teal'], HARVEST['olive']]
for i, row in workflow.reset_index(drop=True).iterrows():
    y = y_positions[i]
    patch = FancyBboxPatch((box_x, y), box_w, box_h, boxstyle='round,pad=0.010,rounding_size=0.02',
                           linewidth=1.15, edgecolor=HARVEST['gray_blue'], facecolor=box_colors[i], alpha=0.28)
    ax_flow.add_patch(patch)
    ax_flow.text(box_x + 0.018, y + 0.053, f"{row['item_key']}", fontsize=12.2, weight='bold', color=TEXT_DARK, va='center')
    ax_flow.text(box_x + 0.112, y + 0.053, row['item_label'], fontsize=12.6, weight='bold', color=TEXT_DARK, va='center')
    ax_flow.text(box_x + 0.112, y + 0.020, wrap_text(row['item_value'], 56), fontsize=9.7, color=TEXT_DARK, va='center')
    if i < len(workflow) - 1:
        next_y = y_positions[i + 1]
        arrow = FancyArrowPatch((box_x + box_w / 2, y - 0.006), (box_x + box_w / 2, next_y + box_h + 0.006),
                                arrowstyle='-|>', mutation_scale=14, linewidth=1.0, color=HARVEST['gray_blue'])
        ax_flow.add_patch(arrow)

ax_cards.set_facecolor(FACE)
ax_cards.set_axis_off()
ax_cards.set_xlim(0, 1)
ax_cards.set_ylim(0, 1)
add_card(ax_cards, 0.03, 0.56, 0.43, 0.33, 'Retained countries', [f'{countries} countries', 'after the', 'country-level threshold'], HARVEST['gray_blue'], title_fontsize=12.2, body_fontsize=10.9, border_width=2.8)
add_card(ax_cards, 0.54, 0.56, 0.43, 0.33, 'Retained rows', [f'{rows:,} phase-level rows', f'from {pre_filter_rows:,} rows', 'after status cleaning'], HARVEST['dark_grass'], title_fontsize=12.2, body_fontsize=10.9, border_width=2.8)
add_card(ax_cards, 0.03, 0.14, 0.43, 0.33, 'Retained capacity', [f'{capacity_tw:.2f} TW', 'utility-scale stock +', 'pipeline composition'], HARVEST['gold'], title_fontsize=12.2, body_fontsize=10.9, border_width=2.8)
add_card(ax_cards, 0.54, 0.14, 0.43, 0.33, 'Frozen mode system', ['4 statuses × 4 size bins', '16 size-status modes'], HARVEST['olive'], title_fontsize=12.2, body_fontsize=10.9, border_width=2.8)

ax_status.set_facecolor(FACE)
status_colors = [HARVEST['dark_grass'], HARVEST['gold'], HARVEST['gray_blue'], HARVEST['olive']]
ax_status.barh(status_retained['Status'], status_retained['rows_utility_sheet'], color=status_colors, edgecolor='none', height=0.78)
for i, row in status_retained.reset_index(drop=True).iterrows():
    ax_status.text(row['rows_utility_sheet'] + 700, i, f"{int(row['rows_utility_sheet']):,}", va='center', fontsize=9.6, color=TEXT_DARK)
ax_status.set_xlabel('Retained phase-level rows', fontsize=10.8, labelpad=2)
ax_status.set_ylabel('')
ax_status.grid(axis='x', color=GRID_SOFT, linewidth=0.8)
ax_status.spines[['top', 'right', 'left']].set_visible(False)
ax_status.tick_params(axis='y', length=0, labelsize=12)
ax_status.tick_params(axis='x', labelsize=10.4)
status_pos = ax_status.get_position()
ax_status.set_position([status_pos.x0, status_pos.y0 + 0.022, status_pos.width, status_pos.height])
ax_status.text(0.0, -0.24, 'Retained statuses: operating, construction, pre-construction, and announced.', transform=ax_status.transAxes, fontsize=9.0, color=TEXT_DARK, ha='left')

fig.subplots_adjust(top=0.97, bottom=0.14, left=0.04, right=0.98)
fig_paths = save_figure(fig, 'F1_workflow_retained_scope')
plt.close(fig)
print(fig_paths)



## Figure 2 — raw vs row-normalized concentration relief


In [ ]:

f2 = F2.copy()
raw_diag = f2.loc[f2['matrix_variant'] == 'raw'].set_index('weight_kind')
norm_diag = f2.loc[f2['matrix_variant'] == 'row_normalized'].set_index('weight_kind')

summary_rows = [
    ('Count | top-5 share', 'count', 'top5_share', HARVEST['gold'], HARVEST['primrose']),
    ('Capacity | top-5 share', 'capacity_mw', 'top5_share', HARVEST['gold'], HARVEST['primrose']),
    ('Count | largest-country share', 'count', 'head_country_share', HARVEST['dark_grass'], HARVEST['olive']),
    ('Capacity | largest-country share', 'capacity_mw', 'head_country_share', HARVEST['dark_grass'], HARVEST['olive']),
]
bar_df = pd.DataFrame([
    {
        'label': label,
        'raw': float(raw_diag.loc[weight_kind, metric]),
        'row_norm': float(norm_diag.loc[weight_kind, metric]),
        'raw_color': raw_color,
        'norm_color': norm_color,
    }
    for label, weight_kind, metric, raw_color, norm_color in summary_rows
])

capacity_raw_graph = build_projection_from_matrix(P2_CAPACITY_RAW, similarity='dot', top_k=6)
capacity_norm_graph = build_projection_from_matrix(P2_CAPACITY_NORM, similarity='cosine', top_k=6)
raw_strength = dict(capacity_raw_graph.degree(weight='weight'))
norm_strength = dict(capacity_norm_graph.degree(weight='weight'))
raw_pos = compute_spread_layout(capacity_raw_graph, raw_strength, seed=42)
norm_pos = compute_spread_layout(capacity_norm_graph, norm_strength, seed=42)
raw_top5 = set(pd.Series(raw_strength).sort_values(ascending=False).head(5).index)
norm_top5 = set(pd.Series(norm_strength).sort_values(ascending=False).head(5).index)

def draw_projection_nodes(ax, graph, strengths, positions, highlight_nodes, title, top_color, other_color, show_labels=True, panel_scale=1.0, label_fontsize=11.5, largest_color=None):
    ax.set_facecolor(FACE)
    ax.set_axis_off()
    ordered_nodes = list(graph.nodes())
    max_strength = max(strengths.values()) if strengths else 1.0
    largest_node = max(strengths, key=strengths.get) if strengths else None
    for node in ordered_nodes:
        x, y = positions[node] * panel_scale
        size = 34 + 460 * np.sqrt(strengths.get(node, 0.0) / max_strength)
        if largest_color is not None and node == largest_node:
            color = largest_color
        else:
            color = top_color if node in highlight_nodes else other_color
        ax.scatter(x, y, s=size, c=color, edgecolors='white', linewidths=1.1, zorder=3)
    if show_labels:
        center = np.mean(np.array([positions[n] * panel_scale for n in ordered_nodes]), axis=0)
        label_nodes = sorted(highlight_nodes, key=lambda n: strengths.get(n, 0.0), reverse=True)
        for rank, node in enumerate(label_nodes):
            x, y = positions[node] * panel_scale
            direction = np.array([x - center[0], y - center[1]], dtype=float)
            norm = float(np.linalg.norm(direction))
            if norm < 1e-9:
                angle = 0.65 * (rank + 1)
                direction = np.array([np.cos(angle), np.sin(angle)])
                norm = 1.0
            direction = direction / norm
            label_radius = 0.17 + 0.02 * rank
            lx, ly = x + direction[0] * label_radius, y + direction[1] * label_radius
            ha = 'left' if direction[0] >= 0 else 'right'
            if direction[1] > 0.25:
                va = 'bottom'
            elif direction[1] < -0.25:
                va = 'top'
            else:
                va = 'center'
            ax.plot([x, lx], [y, ly], color=TEXT_DARK, linewidth=1.0, alpha=0.55, zorder=4)
            ax.text(lx, ly, node, fontsize=label_fontsize, weight='bold', color=TEXT_DARK, ha=ha, va=va,
                    bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.78), zorder=5)
    ax.set_xlim(-1.18, 1.18)
    ax.set_ylim(-1.10, 1.10)
    ax.text(0.01, 1.03, title, transform=ax.transAxes, va='bottom', fontsize=13.0, weight='bold', color=TEXT_DARK, clip_on=False)

fig = plt.figure(figsize=(15.8, 6.3), facecolor=FACE)
gs = gridspec.GridSpec(1, 3, width_ratios=[1.18, 1.0, 1.0], wspace=0.20, figure=fig)
ax_summary = fig.add_subplot(gs[0, 0])
ax_raw = fig.add_subplot(gs[0, 1])
ax_norm = fig.add_subplot(gs[0, 2])

ax_summary.set_facecolor(FACE)
y_positions = np.arange(len(bar_df))
bar_h = 0.28
for i, row in bar_df.iterrows():
    ax_summary.barh(y_positions[i] - bar_h / 2, row['raw'], height=bar_h, color=row['raw_color'], edgecolor='none', zorder=3)
    ax_summary.barh(y_positions[i] + bar_h / 2, row['row_norm'], height=bar_h, color=row['norm_color'], edgecolor='none', zorder=3)
    ax_summary.text(row['raw'] + 0.012, y_positions[i] - bar_h / 2, f"Raw  {row['raw']:.3f}", va='center', fontsize=9.6, color=TEXT_DARK)
    ax_summary.text(row['row_norm'] + 0.012, y_positions[i] + bar_h / 2, f"Row-norm  {row['row_norm']:.3f}", va='center', fontsize=9.6, color=TEXT_DARK)
ax_summary.axhline(1.5, color=GRID_SOFT, linewidth=1.0)
ax_summary.set_yticks(y_positions)
ax_summary.set_yticklabels(bar_df['label'], fontsize=10.6)
ax_summary.invert_yaxis()
ax_summary.set_xlim(0, 0.72)
ax_summary.set_xlabel('Share of projection strength', fontsize=10.6)
ax_summary.grid(axis='x', color=GRID_SOFT, linewidth=0.8)
ax_summary.spines[['top', 'right', 'left']].set_visible(False)
ax_summary.tick_params(axis='y', length=0)
ax_summary.tick_params(axis='x', labelsize=10.6)
ax_summary.text(0.01, 1.03, 'Head-country concentration', transform=ax_summary.transAxes, va='bottom', fontsize=13.0, weight='bold', color=TEXT_DARK, clip_on=False)

draw_projection_nodes(ax_raw, capacity_raw_graph, raw_strength, raw_pos, raw_top5, 'Raw capacity projection', HARVEST['gold'], HARVEST['gray_blue'], show_labels=True, panel_scale=1.0, label_fontsize=11.3, largest_color=HARVEST['dark_grass'])
draw_projection_nodes(ax_norm, capacity_norm_graph, norm_strength, norm_pos, norm_top5, 'Row-normalized capacity projection', HARVEST['primrose'], HARVEST['gray_blue'], show_labels=False, panel_scale=0.90, largest_color=HARVEST['olive'])

legend_items = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=HARVEST['dark_grass'], markeredgecolor='white', markersize=10, label='Largest raw-strength country'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=HARVEST['gold'], markeredgecolor='white', markersize=10, label='Other top-5 raw-strength countries'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=HARVEST['gray_blue'], markeredgecolor='white', markersize=10, label='Other countries'),
]
ax_raw.legend(handles=legend_items, frameon=False, fontsize=9.2, loc='lower left', bbox_to_anchor=(0.02, -0.03), borderaxespad=0.0)
legend_items_norm = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=HARVEST['olive'], markeredgecolor='white', markersize=10, label='Largest row-normalized country'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=HARVEST['primrose'], markeredgecolor='white', markersize=10, label='Other top-5 row-normalized countries'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=HARVEST['gray_blue'], markeredgecolor='white', markersize=10, label='Other countries'),
]
ax_norm.legend(handles=legend_items_norm, frameon=False, fontsize=9.2, loc='lower left', bbox_to_anchor=(0.02, -0.03), borderaxespad=0.0)

fig_paths = save_figure(fig, 'F2_normalization_contrast')
plt.close(fig)
print(fig_paths)



## Figure 3 — capacity-based camps


In [ ]:

capacity_summary = T2.loc[T2['lens'] == 'capacity_main'].sort_values('camp_id').copy()
capacity_map = F3[['Country/Area', 'community']].drop_duplicates().copy()
world_plot, point_df = prepare_map_layers(capacity_map, 'Country/Area', 'community', CAPACITY_COLORS)

fig = plt.figure(figsize=(16.4, 12.8), facecolor='#fbfaf6')
gs = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[3.12, 2.08], hspace=0.04)
ax_map = fig.add_subplot(gs[0, 0])
ax_cards = fig.add_subplot(gs[1, 0])

draw_world_map(ax_map, world_plot, point_df, 'community', CAPACITY_COLORS)

ax_cards.set_facecolor(FACE)
ax_cards.set_axis_off()
ax_cards.set_xlim(0, 1)
ax_cards.set_ylim(0, 1)

left_margin = 0.072
right_margin = 0.072
col_gap = 0.035
col_w = (1 - left_margin - right_margin - 2 * col_gap) / 3
x_positions = [left_margin, left_margin + col_w + col_gap, left_margin + 2 * (col_w + col_gap)]
y_positions = [0.62, 0.21]
card_h = 0.29

capacity_titles = {
    1: '50-500 pre-construction',
    2: '500+ precon/op mixed',
    3: '50-500 announcement',
    4: '1-5 operating',
    5: '500+ announcement',
    6: '5-50 operating',
}

for idx, row in enumerate(capacity_summary.itertuples(index=False)):
    color = CAPACITY_COLORS[int(row.camp_id)]
    col = idx % 3
    row_pos = idx // 3
    lines = [
        f'{int(row.countries)} countries',
        f'Mode: {format_mode_with_share(row.dominant_mode, row.dominant_mode_share)}',
    ]
    add_card(
        ax_cards,
        x_positions[col],
        y_positions[row_pos],
        col_w,
        card_h,
        [f'Camp {int(row.camp_id)}', capacity_titles[int(row.camp_id)]],
        lines,
        color,
        title_fontsize=15.0 if int(row.camp_id) == 2 else 15.6,
        body_fontsize=14.2,
        border_width=6.6,
        body_offset=-0.010 if int(row.camp_id) == 2 else -0.006,
    )

fig.subplots_adjust(left=0.03, right=0.97, top=0.98, bottom=0.05)
fig_paths = save_figure(fig, 'F3_capacity_camps_worldmap')
plt.close(fig)
print(fig_paths)



## Figure 4 — null-model comparison


In [ ]:

null_draws = F4.loc[F4['record_type'] == 'null_draw'].copy()
null_summary = F4.loc[F4['record_type'] == 'summary'].drop_duplicates('weight_kind').set_index('weight_kind')
rng = np.random.default_rng(42)

plot_specs = [
    ('capacity', 'Capacity view', HARVEST['primrose'], HARVEST['gold'], 0.68, 0.79),
    ('count', 'Count view', HARVEST['pale_teal'], HARVEST['gray_blue'], 0.18, 0.29),
]

def simple_kde(draws, grid):
    draws = np.asarray(draws, dtype=float)
    if len(draws) == 0:
        return np.zeros_like(grid)
    if len(draws) == 1:
        bw = 0.006
    else:
        std = np.std(draws, ddof=1)
        bw = max(1.06 * std * (len(draws) ** (-1 / 5)), 0.004)
    diff = (grid[None, :] - draws[:, None]) / bw
    density = np.exp(-0.5 * diff**2).sum(axis=0) / (len(draws) * bw * np.sqrt(2 * np.pi))
    return density

x_left = (0.10, 0.40)
x_right = (0.60, 0.70)
grid_left = np.linspace(x_left[0], x_left[1], 400)

fig = plt.figure(figsize=(14.8, 6.7), facecolor=FACE)
gs = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[4.6, 1.55], wspace=0.05)
ax_left = fig.add_subplot(gs[0, 0])
ax_right = fig.add_subplot(gs[0, 1], sharey=ax_left)
for ax in (ax_left, ax_right):
    ax.set_facecolor(FACE)
    ax.set_ylim(0.02, 1.00)
    ax.set_yticks([])
    ax.grid(axis='x', color=GRID_SOFT, linewidth=0.8)
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.spines['bottom'].set_linewidth(1.6)
    ax.tick_params(axis='x', labelsize=10.0)

ax_left.set_xlim(*x_left)
ax_right.set_xlim(*x_right)
ax_left.set_xticks([0.1, 0.2, 0.3, 0.4])
ax_right.set_xticks([0.6, 0.7])
ax_right.spines['left'].set_visible(False)
ax_right.tick_params(axis='y', left=False, labelleft=False)

for weight_kind, label, cloud_color, obs_color, y, stat_y in plot_specs:
    draws = null_draws.loc[null_draws['weight_kind'] == weight_kind, 'null_modularity'].astype(float).to_numpy()
    summary = null_summary.loc[weight_kind]
    density = simple_kde(draws, grid_left)
    density_height = 0.15 * density / density.max()
    ax_left.fill_between(grid_left, y, y + density_height, color=cloud_color, alpha=0.95, linewidth=0, zorder=1)
    ax_left.plot(grid_left, y + density_height, color=cloud_color, linewidth=1.4, zorder=2)
    jitter = rng.uniform(-0.040, -0.012, size=len(draws))
    ax_left.scatter(draws, np.full_like(draws, y) + jitter, s=18, color=cloud_color, edgecolors='white', linewidths=0.45, alpha=0.98, zorder=3)
    ax_left.hlines(y, x_left[0], x_left[1], color=cloud_color, linewidth=1.8, alpha=0.7, zorder=1)
    ax_right.hlines(y, x_right[0], x_right[1], color=cloud_color, linewidth=1.8, alpha=0.7, zorder=1)
    ax_left.vlines(summary['null_mean'], y - 0.055, y + 0.135, colors='#64748b', linestyles=(0, (3, 2)), linewidth=2.0, zorder=4)
    ax_right.vlines(summary['observed_modularity'], y - 0.060, y + 0.175, colors=obs_color, linewidth=4.2, zorder=5)
    ax_left.text(0.105, y + 0.26, label, fontsize=14.4, weight='bold', color=TEXT_DARK, va='top')
    ax_left.text(0.105, y + 0.20, '40 degree-preserving null replicates', fontsize=9.8, color='#64748b', va='top')
    fig.text(
        0.58,
        stat_y,
        f"Observed = {summary['observed_modularity']:.3f}\nNull mean = {summary['null_mean']:.3f}\nz = {summary['z_score']:.1f}",
        fontsize=10.7,
        color=obs_color,
        ha='center',
        va='center',
        linespacing=1.15,
    )

d = 0.010
kwargs = dict(color='#64748b', clip_on=False, linewidth=1.2)
ax_left.plot((1 - d, 1 + d), (-d, +d), transform=ax_left.transAxes, **kwargs)
ax_left.plot((1 - d, 1 + d), (1 - d, 1 + d), transform=ax_left.transAxes, **kwargs)
ax_right.plot((-d, +d), (-d, +d), transform=ax_right.transAxes, **kwargs)
ax_right.plot((-d, +d), (1 - d, 1 + d), transform=ax_right.transAxes, **kwargs)

fig.supxlabel('Modularity', fontsize=10.6, y=0.035)

fig_paths = save_figure(fig, 'F4_null_model_comparison')
plt.close(fig)
print(fig_paths)



## Figure 5 — count-based higher-order camps


In [ ]:

count_summary = T2.loc[T2['lens'] == 'count_higher_order'].sort_values('camp_id').copy()
count_map = F5[['Country/Area', 'main_text_count_camp']].drop_duplicates().copy()
world_plot, point_df = prepare_map_layers(count_map, 'Country/Area', 'main_text_count_camp', COUNT_COLORS)

fig = plt.figure(figsize=(16.4, 12.4), facecolor='#fbfaf6')
gs = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[3.12, 1.98], hspace=0.04)
ax_map = fig.add_subplot(gs[0, 0])
ax_cards = fig.add_subplot(gs[1, 0])

draw_world_map(ax_map, world_plot, point_df, 'main_text_count_camp', COUNT_COLORS)

ax_cards.set_facecolor(FACE)
ax_cards.set_axis_off()
ax_cards.set_xlim(0, 1)
ax_cards.set_ylim(0, 1)

left_margin = 0.072
right_margin = 0.072
col_gap = 0.035
col_w = (1 - left_margin - right_margin - 2 * col_gap) / 3
x_positions = [left_margin, left_margin + col_w + col_gap, left_margin + 2 * (col_w + col_gap)]
y_positions = [0.62, 0.21]
card_h = 0.29

ecology_labels = {1: 'Ecology A', 2: 'Ecology B', 3: 'Ecology C', 4: 'Ecology D', 5: 'Ecology E'}
ecology_titles = {
    1: 'broad small-operating',
    2: 'mixed small/mid-operating',
    3: 'high-purity small-operating',
    4: 'pipeline-transition',
    5: 'small-operating mixed',
}
ecology_descriptors = {
    1: '1-5 operating',
    2: '5-50 / 1-5 mixed operating',
    3: '1-5 operating',
    4: 'pre-construction / announced mixed\n(5-50-led)',
    5: 'operating 1-5 / 5-50 mixed',
}

for idx, row in enumerate(count_summary.itertuples(index=False)):
    color = COUNT_COLORS[int(row.camp_id)]
    if idx < 3:
        x = x_positions[idx]
        y = y_positions[0]
    else:
        x = x_positions[idx - 3]
        y = y_positions[1]
    if int(row.camp_id) == 4:
        lines = [
            'precon/ann mixed (5-50 led)',
            f"{int(row.countries)} countries",
            f"Source communities: {row.source_communities}",
        ]
    else:
        lines = [
            ecology_descriptors[int(row.camp_id)],
            f"{int(row.countries)} countries",
            f"Source communities: {row.source_communities}",
        ]
    add_card(
        ax_cards,
        x,
        y,
        col_w,
        card_h,
        [ecology_labels[int(row.camp_id)], ecology_titles[int(row.camp_id)]],
        lines,
        color,
        title_fontsize=15.0,
        body_fontsize=13.5,
        border_width=6.6,
        body_offset=0.004 if int(row.camp_id) == 4 else 0.0,
    )

fig.subplots_adjust(left=0.03, right=0.97, top=0.98, bottom=0.05)
fig_paths = save_figure(fig, 'F5_count_higher_order_camps_worldmap')
plt.close(fig)
print(fig_paths)



## Figure 7 — mode distinction audit


In [ ]:

mode_df = F6.copy()
mode_df['combined_class'] = mode_df.apply(combined_mode_class, axis=1)
mode_df['label'] = mode_df['mode'].str.replace('pre-construction', 'pre-constr.', regex=False)
mode_df = mode_df.sort_values(['shared_priority', 'mode'], ascending=[False, True]).reset_index(drop=True)

label_modes = [
    'operating | 1-5',
    'operating | 5-50',
    'announced | 50-500',
    'pre-construction | 50-500',
    'announced | 500+',
    'pre-construction | 5-50',
]
label_vertical = {
    'operating | 1-5': 'below',
    'operating | 5-50': 'below',
    'announced | 50-500': 'above',
    'pre-construction | 50-500': 'below',
    'announced | 500+': 'below',
    'pre-construction | 5-50': 'below',
}
label_df = mode_df.loc[mode_df['mode'].isin(label_modes)].copy()

fig = plt.figure(figsize=(17.6, 7.3), facecolor='#fbfaf6')
gs = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1.30, 0.94], wspace=0.34)
ax_scatter = fig.add_subplot(gs[0, 0])
ax_rank = fig.add_subplot(gs[0, 1])

ax_scatter.set_facecolor(FACE)
ax_rank.set_facecolor(FACE)
for cls, sub in mode_df.groupby('combined_class'):
    ax_scatter.scatter(
        sub['salience_score_count'],
        sub['salience_score_capacity'],
        s=90 + 230 * sub['shared_priority'],
        c=CLASS_COLORS[cls],
        edgecolors='white',
        linewidths=1.2,
        alpha=0.95,
        label=cls,
        zorder=3,
    )

ax_scatter.plot([0.2, 1.02], [0.2, 1.02], linestyle='--', color='#94a3b8', linewidth=1.2, zorder=1)
for row in label_df.itertuples(index=False):
    preferred = label_vertical.get(row.mode, 'above')
    if row.mode != 'announced | 50-500' and row.salience_score_capacity > 0.88:
        preferred = 'below'
    elif row.salience_score_capacity < 0.28:
        preferred = 'above'
    dy = 16 if preferred == 'above' else -16
    va = 'bottom' if preferred == 'above' else 'top'
    if row.salience_score_count < 0.28:
        dx = 2
        ha = 'left'
    elif row.salience_score_count > 0.94:
        dx = -2
        ha = 'right'
    else:
        dx = 0
        ha = 'center'
    ax_scatter.annotate(
        row.label,
        xy=(row.salience_score_count, row.salience_score_capacity),
        xytext=(dx, dy),
        textcoords='offset points',
        fontsize=9.0,
        color='#1f2937',
        ha=ha,
        va=va,
        bbox=dict(boxstyle='round,pad=0.16', facecolor='white', edgecolor='none', alpha=0.82),
        arrowprops=dict(arrowstyle='-', color='#94a3b8', linewidth=0.8),
    )

ax_scatter.set_xlim(0.18, 1.03)
ax_scatter.set_ylim(0.18, 1.03)
ax_scatter.set_xlabel('Count salience score', fontsize=10.6)
ax_scatter.set_ylabel('Capacity salience score', fontsize=10.6)
ax_scatter.grid(color='#d8dee9', linewidth=0.8)
ax_scatter.spines[['top', 'right']].set_visible(False)
ax_scatter.tick_params(axis='both', labelsize=10.0)
legend_handles = [Line2D([0], [0], marker='o', color='w', label=cls, markerfacecolor=color, markersize=10) for cls, color in CLASS_COLORS.items() if cls in mode_df['combined_class'].unique()]
ax_scatter.legend(handles=legend_handles, frameon=False, fontsize=9.2, loc='lower right')

rank_df = mode_df.sort_values('shared_priority', ascending=True).copy()
ax_rank.barh(rank_df['label'], rank_df['shared_priority'], color=rank_df['combined_class'].map(CLASS_COLORS), edgecolor='none')
ax_rank.set_xlabel('Mean cross-view priority', fontsize=10.6)
ax_rank.set_ylabel('')
ax_rank.grid(axis='x', color='#d8dee9', linewidth=0.8)
ax_rank.spines[['top', 'right', 'left']].set_visible(False)
ax_rank.tick_params(axis='x', labelsize=10.0)
ax_rank.tick_params(axis='y', length=0, labelsize=9.2, pad=2)

fig_paths = save_figure(fig, 'F6_mode_distinction_audit')
plt.close(fig)
print(fig_paths)



## Output inventory


In [ ]:

outputs = sorted([path.name for path in FIGURES.glob('*') if path.suffix.lower() == '.png'])
index_rows = []
for stem, title in [
    ('F1_workflow_retained_scope', 'F1 workflow and retained scope'),
    ('F2_normalization_contrast', 'F2 raw vs row-normalized concentration relief'),
    ('F3_capacity_camps_worldmap', 'F3 capacity-based camps'),
    ('F4_null_model_comparison', 'F4 null-model comparison'),
    ('F5_count_higher_order_camps_worldmap', 'F5 count-based higher-order camps'),
    ('F6_mode_distinction_audit', 'F6 mode distinction audit'),
]:
    index_rows.append({'figure_stem': stem, 'title': title, 'png': f'{stem}.png'})
index_df = pd.DataFrame(index_rows)
index_df.to_csv(FIGURES / 'figure_index.csv', index=False)

inventory_lines = ['# Figure Inventory', '']
for row in index_df.itertuples(index=False):
    inventory_lines.append(f"- {row.figure_stem}: {row.title} | png=`{row.png}`")
(FIGURES / 'figure_index.md').write_text('\\n'.join(inventory_lines), encoding='utf-8')

display(index_df)
display(Markdown((FIGURES / 'figure_index.md').read_text(encoding='utf-8')))
